# Étape 03 — Stockage et interrogation (BigQuery)

Projet LB Prod. Ici je charge les données collectées dans l'étape 02 (data/channel_profile.csv et data/videos.csv) dans BigQuery, puis j'écris des requêtes SQL pour en tirer des infos utiles pour LB Prod.

## 1. Import des librairies

Il me faut : pandas (déjà utilisé en étape 02), et google-cloud-bigquery, la librairie officielle pour parler à BigQuery depuis Python. Si elle n'est pas installée : `pip install google-cloud-bigquery`.

Je dois aussi être authentifié auprès de Google Cloud sur cette machine (`gcloud auth application-default login` dans le terminal, une seule fois).

In [1]:
pip install google-cloud-bigquery

Note: you may need to restart the kernel to use updated packages.


## 2. Connexion au projet BigQuery

Je réutilise le projet Google Cloud déjà créé pour le TP : `api-au-tableau-de-bord`. Je crée un client BigQuery pointé sur ce projet.

In [2]:
# on crée un client pour parler à BigQuery
from google.cloud import bigquery

client = bigquery.Client()

## 3. Création du dataset

Je crée un dataset dédié à ce projet (pour ne pas mélanger avec celui de CoinGecko) : `youtube_artiste`. Un dataset BigQuery, c'est l'équivalent d'une "base de données" qui va contenir mes tables.

In [3]:
# on prépare le dataset (nom complet = projet.nom_dataset)
dataset_id = "api-au-tableau-de-bord.youtube_artiste"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "EU"
# exists_ok=True évite une erreur si le dataset existe déjà
client.create_dataset(dataset, exists_ok=True)

Dataset(DatasetReference('api-au-tableau-de-bord', 'youtube_artiste'))

## 4. Chargement de channel_profile.csv

Je charge data/channel_profile.csv dans une table `youtube_artiste.channel_profile`.

In [4]:
import pandas as pd
# on relit le CSV exporté à l'étape 02
channel = pd.read_csv("../data/channel_profile.csv")
table_id = "api-au-tableau-de-bord.youtube_artiste.channel_profile"
# WRITE_TRUNCATE = on remplace toute la table à chaque fois, pas de doublons
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
job = client.load_table_from_dataframe(channel, table_id, job_config=job_config)
job.result()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=api-au-tableau-de-bord, location=EU, id=78faeb3c-28b7-409b-b84e-779d70d0d229>

In [5]:
# même logique pour la table des vidéos
videos = pd.read_csv("../data/videos.csv")
table_id = "api-au-tableau-de-bord.youtube_artiste.videos"
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
job = client.load_table_from_dataframe(videos, table_id, job_config=job_config)
job.result()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=api-au-tableau-de-bord, location=EU, id=d92ed93c-77ce-4ecc-810b-260f0549eb83>

## 6. Vérification du chargement

Je compte le nombre de lignes dans chaque table pour vérifier que tout a bien été chargé (68 vidéos, 1 ligne pour channel_profile).

In [6]:
# on compte les lignes de la table videos pour vérifier le chargement
query = "SELECT COUNT(*) AS nb_lignes FROM `api-au-tableau-de-bord.youtube_artiste.videos`"
result = client.query(query).to_dataframe()
result

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,nb_lignes
0,67


## 7. Requête SQL n°1 — Top 10 des vidéos les plus vues

Utile pour LB Prod : savoir quels titres de Wilson ont le mieux marché, pour s'en inspirer sur les prochaines sorties.

In [7]:
# top 10 des vidéos les plus vues
top_10 = client.query("SELECT * FROM `api-au-tableau-de-bord.youtube_artiste.videos` " \
"ORDER BY view_count DESC LIMIT 10").to_dataframe()

top_10.head(10)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,video_id,title,published_at,view_count,like_count,comment_count,collected_at
0,_rff6xFhb9E,WILSON - VALIDÉ (Clip Officiel),2018-10-19 16:30:01+00:00,37018775,212609,3063,2026-10-14 08:25:13.167446+00:00
1,A7aTBemow4c,WILSON - J'TE LE DIS (Clip Officiel),2020-05-31 15:02:02+00:00,13923659,124327,2932,2026-10-14 08:25:13.167446+00:00
2,DNDTI1xPl3M,WILSON - PIQUÉ (Clip Officiel),2019-06-13 16:33:44+00:00,7077311,85833,2372,2026-10-14 08:25:13.167446+00:00
3,ExqaybpnyKc,WILSON - KIMONO (Clip Officiel),2018-12-14 17:27:55+00:00,3608320,37408,691,2026-10-14 08:25:13.167446+00:00
4,JDwN0h1R7hY,WILSON - OUBLIE TON EX,2023-09-15 16:00:13+00:00,2630024,30430,866,2026-10-14 08:25:13.167446+00:00
5,Ce1o-bG8N90,WILSON - BRISÉ (Clip Officiel),2019-02-15 17:30:00+00:00,1895445,27399,522,2026-10-14 08:25:13.167446+00:00
6,2gBL-xi_1xw,WILSON - On s'en ira (Clip Officiel),2022-04-24 16:00:14+00:00,1567033,21442,694,2026-10-14 08:25:13.167446+00:00
7,ri3dr_VJdlc,WILSON - J'AI DÉJA (Clip Officiel),2019-09-20 16:00:19+00:00,1516575,24182,659,2026-10-14 08:25:13.167446+00:00
8,VLeI1BM0j9c,WILSON - KARABA (Clip Officiel),2021-05-06 16:00:15+00:00,1477838,18225,447,2026-10-14 08:25:13.167446+00:00
9,qql1QNMOHWE,WILSON - RAPPELLE-TOI (Clip Officiel),2021-02-12 17:00:15+00:00,1342778,17062,404,2026-10-14 08:25:13.167446+00:00


## 8. Requête SQL n°2 — Taux d'engagement moyen par vidéo

Taux d'engagement = (likes + commentaires) / vues. Utile pour LB Prod : une vidéo avec beaucoup de vues mais peu d'engagement n'a pas le même intérêt promo qu'une vidéo qui génère beaucoup d'interactions.

In [8]:
# taux d'engagement = (likes + commentaires) / vues, pour chaque vidéo
taux = client.query("""SELECT title, view_count, like_count, comment_count,(like_count + comment_count) / view_count AS taux_engagement
             FROM `api-au-tableau-de-bord.youtube_artiste.videos`
             ORDER BY taux_engagement DESC
             LIMIT 10""")
taux_engagement = taux.to_dataframe()
taux_engagement.head(10)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,title,view_count,like_count,comment_count,taux_engagement
0,🍯😭🎵,3285,273,26,0.091020
1,On va seulement lover sur ce son 🔥🥁,4639,376,16,0.084501
2,24 octobre 2023,8531,695,19,0.083695
3,Alors vos avis sur le son ? 👂🏾,12001,916,61,0.081410
4,Est ce que vous voulez cette version du coup ?...,4616,343,32,0.081239
5,On s’en ira acoustique,7726,571,18,0.076236
6,Une histoire 💎,6938,506,10,0.074373
7,Comment on dit t’es ma femme dans ta langue?,13496,946,53,0.074022
8,On s’en ira acoustique disponible 😊,3192,225,5,0.072055
9,À bon entendeur 🧏🏾‍♂️,2508,170,9,0.071372


## 9. Requête SQL n°3 — Nombre de vidéos et vues totales par année de publication

Utile pour LB Prod : voir l'évolution de la fréquence de sortie et de l'audience de Wilson au fil des années (analyse dans le temps, demandée par l'énoncé).

In [9]:
# nombre de vidéos et vues totales par année de publication
# TIMESTAMP(published_at) : published_at est redevenu du texte en passant par le CSV,
# on le reconvertit en vraie date avant de pouvoir extraire l'année
nb_vues = client.query("""SELECT EXTRACT(YEAR FROM TIMESTAMP(published_at)) AS annee,
             COUNT(*) AS nb_videos,
             SUM(view_count) AS vues_totales
             FROM `api-au-tableau-de-bord.youtube_artiste.videos`
             GROUP BY annee
             ORDER BY annee""")
nb_vues_df = nb_vues.to_dataframe()
nb_vues_df

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,annee,nb_videos,vues_totales
0,2018,3,41582499
1,2019,6,12772132
2,2020,4,16594801
3,2021,11,4821655
4,2022,9,2396547
5,2023,11,5094808
6,2024,4,2415466
7,2025,9,659671
8,2026,10,790337


## Bilan de l'étape 03

Dataset `youtube_artiste` créé dans `api-au-tableau-de-bord`. Tables `channel_profile` (1 ligne) et `videos` (67 lignes) chargées avec `WRITE_TRUNCATE` pour éviter l'accumulation de doublons entre deux exécutions.

Requêtes :
1. Top 10 vues → "VALIDÉ" en tête, 37M vues.
2. Taux d'engagement (likes+commentaires)/vues → les formats courts (stories, extraits) devancent les clips officiels en proportion, malgré un volume de vues bien plus faible.
3. Vidéos/vues par année → 2018-2020 : 13 vidéos, 71M vues. 2021-2026 : 34 vidéos, 16M vues. Publication plus fréquente depuis 2021, portée par vidéo en net recul.

Point technique : `published_at` perd son typage datetime au passage par le CSV, rechargé en STRING dans BigQuery. Contournement en SQL avec `TIMESTAMP(published_at)` avant `EXTRACT`.